# Config

In [24]:
!git config --global --add safe.directory /tmp/Repository/VRID_language_proyect

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [25]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


# 1) Split dataset

In [30]:
from utils.dataset import get_dataset_to_split, split_dataset
import pandas as pd
import numpy as np
import os

#Save data
path = "/tmp/final_project"
filepath = os.path.join(path, "datasets/features.csv")
df=pd.read_csv(filepath)

feat_col = "Interdisciplinario"
df = get_dataset_to_split(df, feat_col)

#Eliminar elementos indefinidos 
df = df[df["Interdisciplinario"]!="INDEFINIDO"]

#Split data and save idx
ids = np.array(df["Código VRID"])
labels = np.array(df["Interdisciplinario"])
savepath = os.path.join(path, "dataSplits/interdiciplinario/train_test_ids_3folds.json")
split_dataset(savepath, ids, labels)

Test size: 193
Fold 0 - Val size: 257
Archivo guardado exitosamente en /tmp/final_project/dataSplits/interdiciplinario/train_test_ids_3folds.json


# 2) Entrenamiento TF-IDF

### Carga y entrenamiento

In [26]:
import json
import pandas as pd

#Ruta de lectura
path = "/tmp/final_project"

#Lectura de index de separacion de conjuntos train/test
filepath=os.path.join(path, "dataSplits/interdiciplinario/train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
filepath=os.path.join(path, "datasets/data_translated.csv")
df = pd.read_csv(filepath)

In [27]:
#################################### ESTO ES PARA TESTEAR DATOS ANTIGUOS #########################
#Ruta de lectura
path2 = "/tmp/data"

#Lectura de index de separacion de conjuntos train/test
filepath=os.path.join(path2, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
#filepath=os.path.join(path2, "data_translated_concat.csv")
#df = pd.read_csv(filepath)


In [18]:
from sklearn.preprocessing import LabelEncoder
from utils.dataset import gen_dataset_select_cols
from models.TIFD import gen_TFID_vectors
import numpy as np

#Columnas a seleccionar para clasificación
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]

#Lectura de codigos VRID Test
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols, 
                                                 test_col="Interdisciplinario")

#Lectura de codigos VRID Train
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols,
                                                      test_col="Interdisciplinario")
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
y_train = [1 if x == "SI" else 0 for x in y_train]
y_test = [1 if x == "SI" else 0 for x in y_test]

#Creacion de vectores TFID
X_train, X_test = gen_TFID_vectors(X_train, X_test)
print(X_train.shape, X_test.shape)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


(771, 16515) (193, 16515)


In [20]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model
from utils.dataset import CvCustom
from collections import Counter

# 1. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    'RandomForestClassifier',
    'XGBClassifier',
    'SVC',
]

# 2. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)
print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))

# 3. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=20
sample_weight_On=True
scoring='f1_macro'
split_idx_path = os.path.join(path, "dataSplits/interdiciplinario/train_test_ids_3folds.json")
#split_idx_path = os.path.join(path2, "train_test_ids_3folds.json")
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode, split_idx_path), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 4. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [0.1, 'l2', 'lbfgs'] before, using random point [0.1, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [0.1, 'l2', 'lbfgs'] before, using random point [10, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [0.1, 'l2', 'lbfgs'] before, using random point [10, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [0.1, 'l2', 'lbfgs'] before, using random point [0.1, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [0.1, 'l2', '

RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.66, 'std_test_score': 0.02}
RandomForestClassifier: {'mean_test_score': 0.64, 'std_test_score': 0.03}
XGBClassifier: {'mean_test_score': 0.61, 'std_test_score': 0.01}
SVC: {'mean_test_score': 0.66, 'std_test_score': 0.03}


In [21]:
from utils.mlflow import eval_model

# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    print(name)
    results, preds=eval_model(model, X_test, y_test, lang_es)
    print(results)

LogisticRegression
{'accuracy': 0.6217616580310881, 'f1_macro': 0.6110697032436163, 'cm': array([[44, 38],
       [35, 76]]), 'precision': 0.6666666666666666, 'recall': 0.6846846846846847, 'f1_es': 0.5779563725650845, 'f1_en': 0.6552552552552553, 'cm_es': array([[13, 28],
       [17, 53]]), 'cm_en': array([[31, 10],
       [18, 23]])}
RandomForestClassifier
{'accuracy': 0.6217616580310881, 'f1_macro': 0.5896367925902193, 'cm': array([[33, 49],
       [24, 87]]), 'precision': 0.6397058823529411, 'recall': 0.7837837837837838, 'f1_es': 0.5488423644112266, 'f1_en': 0.6339285714285715, 'cm_es': array([[ 6, 35],
       [ 8, 62]]), 'cm_en': array([[27, 14],
       [16, 25]])}
XGBClassifier
{'accuracy': 0.5699481865284974, 'f1_macro': 0.5544460823853364, 'cm': array([[37, 45],
       [38, 73]]), 'precision': 0.6186440677966102, 'recall': 0.6576576576576577, 'f1_es': 0.5410244418896185, 'f1_en': 0.5975011155734047, 'cm_es': array([[13, 28],
       [22, 48]]), 'cm_en': array([[24, 17],
       [1

### Guardado de resultados

In [22]:
from utils.save_results import save_models_and_metrics

savepath = os.path.join(path, "output/interdiciplinario/TF_IDF")
save_models_and_metrics(savepath, results_val, models_dicc, X_test, y_test, df_test, save_preds=True, lang_es=df_test["Español"], mode_classification="binary")

📝 Registrando modelo: LogisticRegression
📝 Registrando modelo: RandomForestClassifier


/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:157: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["y_true"]=y_test
/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:158: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["preds"]=preds


📝 Registrando modelo: XGBClassifier
📝 Registrando modelo: SVC


### Inference

In [23]:
from utils.save_results import load_model

model_path = "/tmp/final_project/output/interdiciplinario/TF_IDF/models"
model_path = os.path.join(model_path, "RandomForestClassifier.joblib")



inference_model = load_model(model_path)
pred = inference_model.predict(X_test)


# 3) SPECTER

### Train

In [3]:
import json
import pandas as pd

#Ruta de lectura
path = "/tmp/final_project"

#Lectura de index de separacion de conjuntos train/test
filepath=os.path.join(path, "dataSplits/interdiciplinario/train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
filepath=os.path.join(path, "datasets/data_translated.csv")
df = pd.read_csv(filepath)

In [ ]:
from utils.dataset import gen_dataset_select_cols
from models.specter import embed_texts
import numpy as np

#Columnas a seleccionar para clasificación
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]

#Lectura de codigos VRID Test
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols, 
                                                 test_col="Interdisciplinario")

#Lectura de codigos VRID Train
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols,
                                                      test_col="Interdisciplinario")
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
y_train = [1 if x == "SI" else 0 for x in y_train]
y_test = [1 if x == "SI" else 0 for x in y_test]

# Calcular embeddings
# Parámetros para cargar modelo
BASE_MODEL = "allenai/specter2_base"
ADAPTER_NAME="allenai/specter2_classification"
X_train = embed_texts(X_train, BASE_MODEL, ADAPTER_NAME)
X_test = embed_texts(X_test, BASE_MODEL, ADAPTER_NAME)

print(X_train.shape, X_test.shape)

In [7]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model
from utils.dataset import CvCustom
from collections import Counter

# 1. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    'RandomForestClassifier',
    'XGBClassifier',
    'SVC',
]

# 2. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)
print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))

# 3. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=20
sample_weight_On=True
scoring='f1_macro'
split_idx_path = os.path.join(path, "dataSplits/interdiciplinario/train_test_ids_3folds.json")
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode, split_idx_path), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 4. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [10, 'l2', 'lbfgs'] before, using random point [1.0, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [0.1, 'l2', 'lbfgs'] before, using random point [1.0, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [10, 'l2', 'lbfgs'] before, using random point [10, 'l2', 'lbfgs']
  warnings.warn(


RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.64, 'std_test_score': 0.02}
RandomForestClassifier: {'mean_test_score': 0.63, 'std_test_score': 0.01}
XGBClassifier: {'mean_test_score': 0.62, 'std_test_score': 0.01}
SVC: {'mean_test_score': 0.64, 'std_test_score': 0.02}


In [9]:
from utils.save_results import eval_model

# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    print(name)
    results, preds = eval_model(model, X_test, y_test, lang_es)
    print(results)

LogisticRegression
{'accuracy': 0.5906735751295337, 'f1_macro': 0.5818730289318526, 'cm': array([[43, 39],
       [40, 71]]), 'precision': 0.6454545454545455, 'recall': 0.6396396396396397, 'f1_es': 0.5226654195652033, 'f1_en': 0.6683146067415731, 'cm_es': array([[12, 29],
       [23, 47]]), 'cm_en': array([[31, 10],
       [17, 24]])}
RandomForestClassifier
{'accuracy': 0.6321243523316062, 'f1_macro': 0.6217253278122843, 'cm': array([[45, 37],
       [34, 77]]), 'precision': 0.6754385964912281, 'recall': 0.6936936936936937, 'f1_es': 0.5818476378373408, 'f1_en': 0.6812200956937797, 'cm_es': array([[14, 27],
       [18, 52]]), 'cm_en': array([[31, 10],
       [16, 25]])}
XGBClassifier
{'accuracy': 0.5803108808290155, 'f1_macro': 0.5614743751577884, 'cm': array([[36, 46],
       [35, 76]]), 'precision': 0.6229508196721312, 'recall': 0.6846846846846847, 'f1_es': 0.5779563725650845, 'f1_en': 0.5609756097560976, 'cm_es': array([[13, 28],
       [17, 53]]), 'cm_en': array([[23, 18],
       [1

### Guardado de resultados

In [12]:
import importlib
from utils import save_results

importlib.reload(save_results)

from utils.save_results import save_models_and_metrics

In [13]:
from utils.save_results import save_models_and_metrics

savepath = os.path.join(path, "output/interdiciplinario/SPECTER")
save_models_and_metrics(savepath, results_val, models_dicc, X_test, y_test, df_test, save_preds=True, lang_es=df_test["Español"], mode_classification="binary")

📝 Registrando modelo: LogisticRegression
📝 Registrando modelo: RandomForestClassifier


/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:157: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["y_true"]=y_test
/tmp/Repository/VRID_language_proyect/BERT/utils/save_results.py:158: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["preds"]=preds


📝 Registrando modelo: XGBClassifier
📝 Registrando modelo: SVC


### Inference